# 06: Train the DR grader

Trains one grader (ImageNet-pretrained backbone, 5-class ICDR grade head + binary referable head)
on the EyePACS `train` split and keeps the epoch with the best `val` referable AUROC. Uses the
frozen patient-level splits from the combined manifest; EyeQ quality rejects are left out.
External sites are not touched here.

**Settings** (right-hand panel): Accelerator **GPU T4 x1** (or P100), Internet **On**,
Persistence **Files only**.
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01: Preprocess EyePACS** (the processed image shards)
- *Datasets* → *Your Datasets* → **retinaguard-manifest** (`manifest.parquet`, private)
- To **resume** a stopped run: also the output of this notebook's previous version. The run
  continues after its last finished epoch.

**Run:** set `MODEL` below, then *Save Version* → *Save & Run All (Commit)*. About 7–10 min
per epoch for ConvNeXt-Tiny at 512 px; at most 20 epochs, early stop after 5 without a better
validation AUROC.

**Output** (`/kaggle/working/runs/grader-<model>/`): `best.pt` (weights of the best epoch),
`last.pt` (full state, for resuming), `metrics.jsonl`, `params.json`, `tags.json`,
`config.yaml`, `curves.png`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
MODEL = "convnext_tiny"  # convnext_tiny | efficientnet_b3
OVERRIDES = []  # Hydra overrides, e.g. ["train.batch_size=16", "train.imbalance=sampler"]

In [ ]:
# Kaggle's preinstalled albumentations may predate the 2.x API used for augmentation.
EXTRAS = '"pandera>=0.20" "hydra-core>=1.3" "timm>=1.0" "albumentations>=2.0"'
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" {EXTRAS}

In [ ]:
# --- 1. Inputs, configuration and an earlier run to resume -----------------------
import shutil
import subprocess
from pathlib import Path

import torch

from retinaguard import __version__
from retinaguard.config import load_config
from retinaguard.tracking import RunLogger

INPUT_ROOT = Path("/kaggle/input")
RUNS_DIR = Path("/kaggle/working/runs")
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/eyepacs_*.zip")})
assert shard_dirs, "EyePACS shards not found: attach the output of notebook 01"

cfg = load_config(
    "train/grader",
    [
        f"model={MODEL}",
        f"output_dir={RUNS_DIR}",
        f"data.manifest={manifests[0]}",
        f"data.shard_dirs=[{','.join(str(d) for d in shard_dirs)}]",
        *OVERRIDES,
    ],
)
run_dir = RUNS_DIR / cfg.run_name
earlier = sorted(INPUT_ROOT.glob(f"**/runs/{cfg.run_name}/last.pt"))
if earlier and not (run_dir / "last.pt").exists():
    shutil.copytree(earlier[0].parent, run_dir, dirs_exist_ok=True)
    print("resuming from", earlier[0].parent)

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
RunLogger(run_dir).set_tags({"git_sha": git_sha, "gpu": gpu, "platform": "kaggle"})
print(f"retinaguard {__version__} @ {git_sha[:10]}, torch {torch.__version__}, GPU {gpu}")
print("manifest:", manifests[0])
print("shards:", [str(d) for d in shard_dirs])

In [ ]:
# --- 2. Train (or continue) ----------------------------------------------------
from retinaguard.train.train_grader import train

run_dir = train(cfg)

In [ ]:
# --- 3. Learning curves and the selected epoch ---------------------------------
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.DataFrame(RunLogger(run_dir).read_metrics())
table = metrics.pivot_table(index="step", columns="key", values="value", aggfunc="last")
columns = [
    "train_loss",
    "val_loss",
    "val_referable_auroc",
    "val_referable_auroc_from_grades",
    "val_quadratic_kappa",
    "epoch_minutes",
]
print(table[columns].round(4).to_string())
best = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
print(f"best epoch {best['epoch']}: val referable AUROC {best['val_referable_auroc']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
table[["train_loss", "val_loss"]].plot(ax=axes[0], title="loss")
table[["val_referable_auroc", "val_referable_auroc_from_grades"]].plot(
    ax=axes[1], title="val referable AUROC"
)
table[["val_quadratic_kappa"]].plot(ax=axes[2], title="val quadratic kappa")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
fig.savefig(run_dir / "curves.png", dpi=80)
plt.show()

**After the version finishes:** send the table and best epoch printed by cell 3. Keep the
version: its output is the input for evaluation (and for resuming). Do not download the `.pt`
files into the repository.